In [1]:
using Plots, Distributions, JLD, DSP, Parameters, Match, SymDict, NumericalIntegration, Random, MAT, Pkg, CSV, DataFrames, XLSX, Pkg, JLD2, Colors, JLD2,  Makie, CairoMakie, Optim, Serialization, JuMP, Ipopt, ForwardDiff, LinearAlgebra

include("creation_model.jl")

spikes_from_aps

In [26]:
function run_sim_inh(i,inh_input)

    full_input = commoninput 
    vd, vs, _ = ode4(tspan, 0.,0., i, pool_state_params[i], full_input,inh_input, MNs)  
     
    spikes = spikes_from_aps(vs, i, MNs)       
    spike_arr = reshape(spikes, 1, length(spikes))
    vs_arr[i,:] = vs
    return spike_arr, vs_arr
end

run_sim_inh (generic function with 1 method)

In [2]:
function run_sim(i,factor)
    full_input = commoninput 
    vd, vs, _ = ode4(tspan, 0.,0., i, pool_state_params[i], full_input,full_input*factor, MNs)       
    spikes = spikes_from_aps(vs, i, MNs)       
    spike_arr = reshape(spikes, 1, length(spikes))
    vs_arr[i,:] = vs
    return spike_arr, vs_arr
end

run_sim (generic function with 1 method)

In [8]:
#### INITIALISATION OF PARAMETERS

nr_s = 10
rheobase_S = LinRange(3.5,6.5,nr_s) .+ rand(Normal(0.0,0.05),nr_s)  
rs_S = LinRange(38.75,41.25,nr_s)./10000 
ls_S = LinRange(77.5,82.5,nr_s)./10000 
rms_S = LinRange(1.15, 1.05,nr_s).*1000
rd_S = LinRange(20.75,31.25,nr_s)./10000 
ld_S = LinRange(5500.,6800.,nr_s)./10000 
rmd_S = LinRange(14.4,10.70,nr_s).*1000 

max_gna = 30.0 
max_gkf_S = ones(nr_s)*4 
max_gks_S =LinRange(16,25,nr_s) 
max_gca_S = LinRange(0.038,0.029,nr_s) 

αm = 22.0; βm = 13.0
αh_S = ones(nr_s)*0.5
βh_S = ones(nr_s)*4
αn_S = ones(nr_s)*1.5
βn_S = ones(nr_s)*0.1
αq_S = ones(nr_s)*1.5
βq_S = ones(nr_s)*0.025
αp = 0.008
βp= LinRange(0.014,0.016,nr_s)
βp_S = ones(nr_s)*0.014

Vthca_S = LinRange(2.5,3.0,nr_s) 
Apeak_S = LinRange(0.103,0.123,nr_s)
tpeak_S = LinRange(110.,100.,nr_s) 
tetanic_S = LinRange(0.393,0.491,nr_s);

amplitude=0.4
tr_length=28000
half_triang = div(tr_length, 2)  
flat_line = fill(-0.1, 1000)
up = range(-0.1, amplitude, length = half_triang + 1)  
down = range(amplitude, -0.1, length = tr_length - half_triang + 1)  
tr = vcat(flat_line, up[2:end], down[2:end], flat_line)
sim_length=30000

γh = zeros(nr_s);


IF= LinRange(1.,1.,nr_s);  
ri = 70.;
cm = 1.; 
MN_list=[];

Vth_S = zeros(nr_s); gls_S = zeros(nr_s); gld_S = zeros(nr_s); gc_S = zeros(nr_s); gna_S = zeros(nr_s); gkf_S = zeros(nr_s); gks_S = zeros(nr_s); gca_S = zeros(nr_s); rn_S = zeros(nr_s); cs_S = zeros(nr_s)  ; cd_S = zeros(nr_s)
for i in 1:nr_s 
    gls_S[i] = ((2*π*rs_S[i]*ls_S[i])/rms_S[i])*1e6  
    gld_S[i] = ((2*π*rd_S[i]*ld_S[i])/rmd_S[i])*1e6 
    gc_S[i] = ((2/(((ri*ld_S[i])/(π*rd_S[i]^2))+((ri*ls_S[i])/(π*rs_S[i]^2)))))*1e6    
    rn_S[i] = 1/(gls_S[i]+((gld_S[i]*gc_S[i])/(gld_S[i]+gc_S[i])))    
    Vth_S[i] = round(rn_S[i]*rheobase_S[i],digits=2)   
    cs_S[i] = 2*π*ls_S[i]*rs_S[i]*cm*1e3  
    cd_S[i] = 2*π*ld_S[i]*rd_S[i]*cm*1e3   
    gna_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gna*1e3          
    gkf_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gkf_S[i]*1e3     
    gks_S[i] = 2*π*ls_S[i]*rs_S[i]*max_gks_S[i]*1e3     
    gca_S[i] = 2*π*ld_S[i]*rd_S[i]*max_gca_S[i]*1e3     

    S_MN = Dict(:Cs => cs_S[i], :Cd => cd_S[i], :Irh => rheobase_S[i], :gc => gc_S[i], :gNa => gna_S[i], :gKs => gks_S[i], :gKf => gkf_S[i], :gCa => gca_S[i], :gld => gld_S[i], :gls => gls_S[i], :αp => αp, :βp => βp_S[i], 
        :αq => αq_S[i], :βq => βq_S[i], :αm => αm, :βm => βm, :αh => αh_S[i], :βh => βh_S[i], :αn => αn_S[i], :βn => βn_S[i],:αp => αp, :βp => βp[i],  :Vth => Vth_S[i], :VthCa => Vthca_S[i], :Rn => rn_S[i],
        :Tetanic => tetanic_S[i], :Apeak => Apeak_S[i],:IF => IF[i], :tpeak => tpeak_S[i], :γh=>γh[i] , "type" => "S Type")
    
    MN_list = [MN_list;S_MN];

end
MNs = MN_list;

EL=0; ENa=120; ECa=140; EK=-10; Ee=70; Ei=-16 
Ge=0.6; Gi=0.5 ; 
Vd0 = 0.; 
Vs0 = 0.;
rp=5.0; 
step = 0.05;
_int(num) = floor(Int, num);
pd=0.6; 

tspan = 0:0.05:sim_length; 
m=0.0;h=1.0;n=0.0;q=0.0;p=0.0;m0=0.0;h0=1.0;n0=0.0;q0=0.0;p0=0.0; t0=-1.0; t1=-1.0; SPIKE0=0; SPIKE1=0
pool_state_params = fill((m, h, n, q, p, m0, h0, n0, q0, p0, t0, t1, SPIKE0, SPIKE1), nr_s) 
sim_time = _int(sim_length/0.05)  
spike_arr = zeros((nr_s, sim_time)) 
vs_arr = zeros((nr_s, sim_time)); 
commoninput = tr; 
full_input = zeros((nr_s, sim_length))
independantnoise = zeros((nr_s, sim_length))





10×30000 Matrix{Float64}:
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0

In [32]:
#########################################################################################
################## CHANGING VALUES OF NEUROMOD AND CONSTANT INHIBITION ##################
#########################################################################################

vector_γ=  LinRange(0.,1.,20)   
vector_inh=  LinRange(-1.,1.,20)
hysteresis_for_each_inh= zeros(20,20)
n=9

for j in 1:20
    for k in 1:20
        
        γh= vector_γ[j] 
        factor= vector_inh[k]        
        inh_input=fill(factor*amplitude, sim_length)
     

        
        MN_list[n][:γh] = γh

        MNs= MN_list

        
        spike_array, vs_array = run_sim_inh(n, inh_input);


        amplitude=0.4;
        sim_length=560000 ;
        half_triang = div(sim_length, 2);  
        flat_line = fill(-0.1, 20000);
        up = range(-0.1, amplitude, length = half_triang + 1)  ;
        down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
        tr = vcat(flat_line, up[2:end], down[2:end], flat_line);
        equiv_force_profile= (tr/maximum(tr))*nr_s

        
        
        if any(spike_array[1, :] .== 1)
            
            value_1= equiv_force_profile[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
            value_2=  equiv_force_profile[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold

            hysteresis_for_each_inh[j,k]= ((value_2/value_1).-1)[1]
        else
            hysteresis_for_each_inh[j,k]=100
        end

    end

end


In [ ]:
################################################################################
################## CHANGING VALUES OF NEUROMOD AND INHIBITION ##################
################################################################################

vector_γ=  LinRange(0.,1.,20)   
vector_inh=  LinRange(-1.,1.,20)
hysteresis_for_each_inh= zeros(20,20)
n=5

for j in 1:20
    for k in 1:20
        
        γh= vector_γ[j] 
        print(γh)
        factor= vector_inh[k]        
        
        MN_list[n][:γh] = γh

        MNs= MN_list

        spike_array, vs_array = run_sim(n, factor);


        amplitude=0.4;
        sim_length=560000 ;
        half_triang = div(sim_length, 2);  
        flat_line = fill(-0.1, 20000);
        up = range(-0.1, amplitude, length = half_triang + 1)  ;
        down = range(amplitude, -0.1, length = sim_length - half_triang + 1);  
        tr = vcat(flat_line, up[2:end], down[2:end], flat_line);
        equiv_force_profile= (tr/maximum(tr))*nr_s

        if any(spike_array[1, :] .== 1)
            
            value_1= equiv_force_profile[findfirst(==(1),spike_array[1,:]),:] #force at the recruitment threshold
            value_2=  equiv_force_profile[findlast(==(1),spike_array[1,:]),:] #force at the derecruitment threshold

            hysteresis_for_each_inh[j,k]= ((value_2/value_1).-1)[1]
        else
            hysteresis_for_each_inh[j,k]=100
        end

    end

end


In [1]:

hysteresis_for_each_inh_modified = copy(hysteresis_for_each_inh)  
hysteresis_for_each_inh_modified[hysteresis_for_each_inh .> 99] .= NaN

Plots.heatmap(hysteresis_for_each_inh_modified,color=:inferno, aspect_ratio=1, grid=false, xticks=false, yticks=false, xlabel="", ylabel="", frame=:box)              


UndefVarError: UndefVarError: hysteresis_for_each_inh not defined